In [1]:
import kagglehub
import tensorflow as tf
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
import keras

In [2]:
# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset


In [3]:
base_dir='/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'

train_dir = os.path.join(base_dir, 'Training')
validation_dir = os.path.join(base_dir, 'Testing')

train_glioma_dir = os.path.join(train_dir, 'glioma')
train_meningioma_dir = os.path.join(train_dir, 'meningioma')
train_pituitary_dir = os.path.join(train_dir, 'pituitary')
train_notumor_dir = os.path.join(train_dir, 'notumor')

validation_glioma_dir = os.path.join(validation_dir, 'glioma')
validation_meningioma_dir = os.path.join(validation_dir, 'meningioma')
validation_pituitary_dir = os.path.join(validation_dir, 'pituitary')
validation_notumor_dir = os.path.join(validation_dir, 'notumor')

In [4]:
image_size = (128,128)
Batch_size = 32

In [5]:
def load_images_and_labels(directory , label):
    images = []
    labels = []

    image_files = [f for f in os.listdir(directory) if f.lower().endswith(('.png' , '.jpg' , '.jpeg'))]
    for filename in image_files:
        image_path = os.path.join(directory , filename)

        image = cv2.imread(image_path)
        image = cv2.resize(image, image_size)

        images.append(image)
        labels.append(label)
    return images , labels

In [6]:
x_train_glioma , y_train_glioma = load_images_and_labels( train_glioma_dir , 0 )
x_train_meningioma , y_train_meningioma = load_images_and_labels( train_meningioma_dir , 1 )
x_train_pituitary , y_train_pituitary = load_images_and_labels( train_pituitary_dir , 2 )
x_train_notumor , y_train_notumor = load_images_and_labels( train_notumor_dir , 3 )

In [7]:
x_validation_glioma , y_validation_glioma = load_images_and_labels( validation_glioma_dir , 0 )
x_validation_meningioma , y_validation_meningioma = load_images_and_labels( validation_meningioma_dir , 1 )
x_validation_pituitary , y_validation_pituitary = load_images_and_labels( validation_pituitary_dir , 2 )
x_validation_notumor , y_validation_notumor = load_images_and_labels( validation_notumor_dir , 3 )

In [8]:
x_train = np.array(x_train_glioma + x_train_meningioma + x_train_pituitary + x_train_notumor)
y_train = np.array(y_train_glioma + y_train_meningioma + y_train_pituitary + y_train_notumor)

x_validation = np.array(x_validation_glioma + x_validation_meningioma + x_validation_pituitary + x_validation_notumor)
y_validation = np.array(y_validation_glioma + y_validation_meningioma + y_validation_pituitary + y_validation_notumor)

In [9]:
def shuffle(images , labels):
    combined = list(zip(images , labels))
    np.random.shuffle(combined)
    shuffled_images , shuffled_labels = zip(*combined)
    return np.array(shuffled_images) , np.array(shuffled_labels)

In [10]:
x_train , y_train = shuffle(x_train , y_train)
x_validation , y_validation = shuffle(x_validation , y_validation)

In [11]:
x_train = x_train.astype('float32') / 255.0
x_validation = x_validation.astype('float32') / 255.0

In [12]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(128, 128 , 3)),


    
  

    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),


    tf.keras.layers.Dense(4, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-08-13 08:10:38.066456: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 49152)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    12,583,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,624,580 (48.16 MB)

 Trainable params: 12,624,580 (48.16 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
opt = tf.keras.optimizers.Adam(learning_rate=0.00001)

In [14]:
callback = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=False,
    start_from_epoch=0,
)

In [15]:
model.compile(optimizer=opt,
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

if len(x_train) > 0 and len(y_train) > 0 :
  history = model.fit(x_train , y_train ,
                      epochs=55,
                      batch_size=Batch_size,
                      callbacks=[callback],
                      validation_data=(x_validation, y_validation))


Epoch 1/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 25s 135ms/step - accuracy: 0.3196 - loss: 1.3565 - val_accuracy: 0.4588 - val_loss: 1.2878
Epoch 2/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 23s 132ms/step - accuracy: 0.3571 - loss: 1.3016 - val_accuracy: 0.4850 - val_loss: 1.2490
Epoch 3/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 23s 132ms/step - accuracy: 0.3991 - loss: 1.2616 - val_accuracy: 0.5962 - val_loss: 1.1802
Epoch 4/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 23s 132ms/step - accuracy: 0.4187 - loss: 1.2246 - val_accuracy: 0.5781 - val_loss: 1.1707
Epoch 5/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 21s 120ms/step - accuracy: 0.4620 - loss: 1.1854 - val_accuracy: 0.6275 - val_loss: 1.1160
Epoch 6/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 22s 124ms/step - accuracy: 0.4668 - loss: 1.1663 - val_accuracy: 0.6275 - val_loss: 1.1055
Epoch 7/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 21s 122ms/step - accuracy: 0.5038 - loss: 1.1233 - val_accuracy: 0.6281 - val_loss: 1.0540
Epoch 8/55
175/175 ━━━━━━━━━━━━━━━━━━━━ 22s 123ms/step - accuracy: 0.5129 - loss: 1

In [16]:
model.save("model_v2.keras")